In [1]:
# =====================================================
# PROJECT PATH SETUP
# =====================================================

import sys
from pathlib import Path

PROJECT_ROOT = (
    Path.cwd()
    .parents[1]
)

SRC_DIR = (
    PROJECT_ROOT /
    "src"
)

if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))


# =====================================================
# STANDARD LIBRARY
# =====================================================

import time
from datetime import datetime
import traceback



# =====================================================
# THIRD-PARTY LIBRARIES
# =====================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import wandb

from wandb.integration.keras import (
    WandbMetricsLogger
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)


# =====================================================
# FMA SMALL CONFIGURATION
# =====================================================

from config import (
    DATASET_NAME,
    MODEL_NAME,
    CLASS_NAMES,
    NUM_CLASSES,
    GENRE_TO_INT,
    IMG_SIZE,
    BATCH_SIZE,
    EPOCHS,
    LEARNING_RATES,
    SEEDS,
    SEEDS_TEST,
    DURATIONS,
    DURATIONS_TEST,
    RUN_MODE,
    PATIENCE,
    MODEL_CATEGORIES,
    PRETRAINED_MODELS,
    WANDB_PROJECT,
    STUDY_NAME
)

ACTIVE_IMG_SIZE = (
    IMG_SIZE[
        MODEL_NAME
    ]
)

# =====================================================
# FMA SMALL UTILITIES
# =====================================================

from fma_small.utils.paths import (

    FMA_SPLIT_FILE,

    FMA_INVENTORY_FILE,

    FMA_SPECTROGRAM_DIR,

    get_output_paths
)

from fma_small.utils.model_factory import (
    build_model
)

from fma_small.utils.evaluation import (
    evaluate_model
)


# =====================================================
# TENSORFLOW
# =====================================================

print(
    f"TensorFlow Version: "
    f"{tf.__version__}"
)


# =====================================================
# SANITY CHECKS
# =====================================================

print("\n=== STUDY CONFIGURATION ===")

print(
    f"Study: {STUDY_NAME}"
)

print(
    f"Dataset: {DATASET_NAME}"
)

print(
    f"Model: {MODEL_NAME}"
)

print(
    f"Classes: {NUM_CLASSES}"
)

print(
    f"Image Size: {ACTIVE_IMG_SIZE}"
)

print(
    f"Batch Size: {BATCH_SIZE}"
)

print(
    f"Epochs: {EPOCHS}"
)

print(
    f"Seeds: {SEEDS}"
)

print(
    f"Durations: {DURATIONS}"
)

print(
    f"Learning Rate: "
    f"{LEARNING_RATES[MODEL_NAME]}"
)


# =====================================================
# DATASET VALIDATION
# =====================================================

print("\n=== DATA ASSETS ===")

print(
    f"Inventory Exists: "
    f"{FMA_INVENTORY_FILE.exists()}"
)

print(
    f"Split File Exists: "
    f"{FMA_SPLIT_FILE.exists()}"
)

print(
    f"Spectrograms Exist: "
    f"{FMA_SPECTROGRAM_DIR.exists()}"
)


# =====================================================
# WANDB VALIDATION
# =====================================================

print("\n=== WANDB ===")

print(
    f"Project: {WANDB_PROJECT}"
)

print(
    "WandB will be initialized "
    "during experiment execution."
)

SyntaxError: unmatched '}' (model_factory.py, line 38)

In [ ]:
import sys

print(sys.executable)


In [ ]:
split_df = pd.read_csv(
    FMA_SPLIT_FILE
)

print(
    split_df["split"]
    .value_counts()
)

print(
    split_df.groupby(
        ["split", "genre"]
    ).size()
)

In [ ]:
def build_file_lists(
    duration,
    split_name,
    split_df
):

    subset = split_df[
        split_df["split"]
        == split_name
    ]

    image_paths = []
    labels = []

    missing_count = 0

    for _, row in subset.iterrows():

        image_path = (
            FMA_SPECTROGRAM_DIR
            / duration
            / row["genre"]
            / f"{row['track_id']}.png"
        )

        if image_path.exists():

            image_paths.append(
                str(image_path)
            )

            labels.append(
                GENRE_TO_INT[
                    row["genre"].strip().lower()
                ]
            )

        else:

            missing_count += 1

    print(
        f"{split_name}: "
        f"{len(image_paths)} images "
        f"({missing_count} missing)"
    )

    return (
        image_paths,
        labels
    )

In [ ]:
def parse_image(
        image_path,
        label):
    
    label = tf.cast(
        label,
        tf.int32
    )
    image = tf.io.read_file(
        image_path
    )

    image = tf.image.decode_png(
        image,
        channels=3
    )

    image = tf.image.resize(
        image,
        ACTIVE_IMG_SIZE
    )

    image = (
        tf.cast(
            image,
            tf.float32
        )
        / 255.0
    )

    return image, label


def make_dataset(
        paths,
        labels,
        training=False):

    ds = (
        tf.data.Dataset
        .from_tensor_slices(
            (paths, labels)
        )
    )

    ds = ds.map(
        parse_image,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if training:

        ds = ds.shuffle(
            buffer_size=len(paths),
            reshuffle_each_iteration=True
        )

    ds = ds.batch(
        BATCH_SIZE
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


In [ ]:
train_paths, train_labels = (
    build_file_lists(
        DURATIONS_TEST[0],
        "train",
        split_df
    )
)

train_ds = make_dataset(
    train_paths,
    train_labels,
    training=True
)

print(train_ds)

In [ ]:
learning_rate = (
    LEARNING_RATES[
        MODEL_NAME
    ]
)

model = build_model(

    model_name=MODEL_NAME,

    input_shape=ACTIVE_IMG_SIZE + (3,),

    num_classes=NUM_CLASSES,

    learning_rate=learning_rate
)

model.summary()

print(
    f"\nParameter Count: "
    f"{model.count_params():,}"
)

print(
    f"Model Category: "
    f"{MODEL_CATEGORIES[MODEL_NAME]}"
)

print(
    f"Pretrained: "
    f"{PRETRAINED_MODELS[MODEL_NAME]}"
)

In [ ]:
# =====================================================
# EXPERIMENT LOOP
# =====================================================

import time

paths = get_output_paths(
    MODEL_NAME
)

all_summary_rows = []


# =====================================================
# ACTIVE CONFIGURATION
# =====================================================

ACTIVE_SEEDS = (
    SEEDS_TEST
    if RUN_MODE == "test"
    else SEEDS
)

ACTIVE_DURATIONS = (
    DURATIONS_TEST
    if RUN_MODE == "test"
    else DURATIONS
)

print("\n=== ACTIVE RUN CONFIGURATION ===")

print(
    f"Run Mode: {RUN_MODE}"
)

print(
    f"Seeds: {ACTIVE_SEEDS}"
)

print(
    f"Durations: {ACTIVE_DURATIONS}"
)


# =====================================================
# REPRODUCIBILITY
# =====================================================

def set_seed(seed):

    np.random.seed(seed)

    tf.random.set_seed(seed)


# =====================================================
# RUN ONE EXPERIMENT
# =====================================================

def run_experiment(
    seed,
    duration
):

    try:

        set_seed(seed)

        print("\n" + "=" * 60)

        print(
            f"Model: {MODEL_NAME}"
        )

        print(
            f"Seed: {seed}"
        )

        print(
            f"Duration: {duration}"
        )

        print("=" * 60)

        # ------------------------------------------
        # Build datasets
        # ------------------------------------------

        train_files, train_labels = (
            build_file_lists(
                duration,
                "train",
                split_df
            )
        )

        val_files, val_labels = (
            build_file_lists(
                duration,
                "val",
                split_df
            )
        )

        test_files, test_labels = (
            build_file_lists(
                duration,
                "test",
                split_df
            )
        )

        train_ds = make_dataset(
            train_files,
            train_labels,
            training=True
        )

        val_ds = make_dataset(
            val_files,
            val_labels
        )

        test_ds = make_dataset(
            test_files,
            test_labels
        )

        # ------------------------------------------
        # Build model
        # ------------------------------------------

        learning_rate = (
            LEARNING_RATES[
                MODEL_NAME
            ]
        )

        model = build_model(
            model_name=MODEL_NAME,
            input_shape=ACTIVE_IMG_SIZE + (3,),
            num_classes=NUM_CLASSES,
            learning_rate=learning_rate
        )

        parameter_count = (
            model.count_params()
        )

        run_name = (
            f"{MODEL_NAME}_"
            f"{duration}_"
            f"s{seed}"
        )

        # ------------------------------------------
        # WandB Run
        # ------------------------------------------

        wandb.init(

            project=WANDB_PROJECT,

            name=run_name,

            config={

                "study":
                    STUDY_NAME,

                "dataset":
                    DATASET_NAME,

                "model":
                    MODEL_NAME,

                "model_category":
                    MODEL_CATEGORIES[
                        MODEL_NAME
                    ],

                "pretrained":
                    PRETRAINED_MODELS[
                        MODEL_NAME
                    ],

                "seed":
                    seed,

                "duration":
                    duration,

                "learning_rate":
                    learning_rate,

                "batch_size":
                    BATCH_SIZE,

                "epochs":
                    EPOCHS,

                "image_size":
                    ACTIVE_IMG_SIZE,

                "num_classes":
                    NUM_CLASSES,

                "parameter_count":
                    parameter_count,

                "train_samples":
                    len(train_files),

                "val_samples":
                    len(val_files),

                "test_samples":
                    len(test_files),

                "run_timestamp":
                    datetime.now().isoformat()
            }
        )

        # ------------------------------------------
        # Save architecture metadata
        # ------------------------------------------

        metadata_path = (
            paths["metadata"]
            /
            f"{MODEL_NAME}_metadata.csv"
        )

        if not metadata_path.exists():

            pd.DataFrame([{

                "model":
                    MODEL_NAME,

                "model_category":
                    MODEL_CATEGORIES[
                        MODEL_NAME
                    ],

                "pretrained":
                    PRETRAINED_MODELS[
                        MODEL_NAME
                    ],

                "parameter_count":
                    parameter_count,

                "input_height":
                    ACTIVE_IMG_SIZE[0],

                "input_width":
                    ACTIVE_IMG_SIZE[1],

                "channels":
                    3,

                "num_classes":
                    NUM_CLASSES,

                "learning_rate":
                    learning_rate

            }]).to_csv(
                metadata_path,
                index=False
            )

        # ------------------------------------------
        # Callbacks
        # ------------------------------------------

        checkpoint_path = (
            paths["checkpoints"]
            /
            f"{MODEL_NAME}_{duration}_seed_{seed}.keras"
        )

        early_stop = EarlyStopping(

            monitor="val_accuracy",

            mode="max",

            patience=PATIENCE,

            restore_best_weights=True,

            verbose=1
        )

        reduce_lr = (
            ReduceLROnPlateau(

                monitor="val_accuracy",

                mode="max",

                factor=0.5,

                patience=5,

                min_lr=1e-6,

                verbose=1
            )
        )

        checkpoint = (
            ModelCheckpoint(

                filepath=checkpoint_path,

                monitor="val_accuracy",

                mode="max",

                save_best_only=True,

                verbose=1
            )
        )

        # ------------------------------------------
        # Train
        # ------------------------------------------

        start_time = time.time()

        history = model.fit(

            train_ds,

            validation_data=val_ds,

            epochs=EPOCHS,

            callbacks=[

                early_stop,

                reduce_lr,

                checkpoint,

                WandbMetricsLogger()
            ],

            verbose=1
        )

        training_seconds = (
            time.time()
            -
            start_time
        )

        # ------------------------------------------
        # Best Epoch
        # ------------------------------------------

        best_epoch = (

            np.argmax(
                history.history[
                    "val_accuracy"
                ]
            )
            + 1
        )

        epochs_completed = len(
            history.history[
                "loss"
            ]
        )

        # ------------------------------------------
        # Save Training History
        # ------------------------------------------

        history_df = pd.DataFrame(
            history.history
        )

        history_df["epoch"] = np.arange(
            1,
            len(history_df) + 1
        )

        history_df["model"] = MODEL_NAME
        history_df["duration"] = duration
        history_df["seed"] = seed

        history_df.to_csv(

            paths["histories"]
            /
            f"training_history_{duration}_seed_{seed}.csv",

            index=False
        )

        # ------------------------------------------
        # Evaluate
        # ------------------------------------------

        metrics = evaluate_model(

            model=model,

            test_ds=test_ds,

            seed=seed,

            duration=duration,

            best_epoch=best_epoch,

            num_classes=NUM_CLASSES,

            class_names=CLASS_NAMES,

            log_to_wandb=True
        )

        # ------------------------------------------
        # Save confusion matrix
        # ------------------------------------------

        np.save(

            paths[
                "confusion_matrices"
            ]
            /
            f"confusion_matrix_{duration}_seed_{seed}.npy",

            metrics[
                "confusion_matrix"
            ]
        )

        # ------------------------------------------
        # Summary row
        # ------------------------------------------

        summary_row = {

            "run_name":
                run_name,

            "model":
                MODEL_NAME,

            "seed":
                seed,

            "duration":
                duration,

            "model_category":
                MODEL_CATEGORIES[
                    MODEL_NAME
                ],

            "pretrained":
                PRETRAINED_MODELS[
                    MODEL_NAME
                ],

            "parameter_count":
                parameter_count,

            "learning_rate":
                learning_rate,

            "epochs_completed":
                epochs_completed,

            "best_epoch":
                best_epoch,

            "training_seconds":
                training_seconds,

            "best_val_accuracy":
                float(
                    np.max(
                        history.history[
                            "val_accuracy"
                        ]
                    )
                ),

            "best_val_loss":
                float(
                    np.min(
                        history.history[
                            "val_loss"
                        ]
                    )
                ),

            "test_accuracy":
                metrics["accuracy"],

            "test_balanced_accuracy":
                metrics[
                    "balanced_accuracy"
                ],

            "test_precision":
                metrics["precision"],

            "test_recall":
                metrics["recall"],

            "test_f1":
                metrics["f1"],

            "test_top2_accuracy":
                metrics[
                    "top2_accuracy"
                ],

            "test_top3_accuracy":
                metrics[
                    "top3_accuracy"
                ],

            "test_kappa":
                metrics["kappa"]
        }

        # ------------------------------------------
        # Save run summary immediately
        # ------------------------------------------

        pd.DataFrame(
            [summary_row]
        ).to_csv(

            paths["duration_results"]
            /
            f"summary_{duration}_seed_{seed}.csv",

            index=False
        )

        wandb.log(
            summary_row
        )

        return summary_row

    finally:

        if wandb.run is not None:
            wandb.finish()


# =====================================================
# EXECUTE EXPERIMENTS
# =====================================================

for seed in ACTIVE_SEEDS:

    for duration in ACTIVE_DURATIONS:

        try:

            summary_row = run_experiment(
                seed,
                duration
            )

            all_summary_rows.append(
                summary_row
            )
        except Exception as e:

            print("\nFAILED")

            print(
                f"Model={MODEL_NAME}"
            )

            print(
                f"Seed={seed}"
            )

            print(
                f"Duration={duration}"
            )

            traceback.print_exc()

            continue


# =====================================================
# SAVE GLOBAL SUMMARY
# =====================================================

summary_df = pd.DataFrame(
    all_summary_rows
)

summary_df.to_csv(

    paths["duration_results"]
    /
    "experiment_summary_all.csv",

    index=False
)

print(
    "\nSaved experiment_summary_all.csv"
)

display(
    summary_df
)

In [ ]:
split_df.groupby(
    ["split", "genre"]
).size()